# 15 — Multi-Agent System

This notebook demonstrates NeuroForge's **multi-agent architecture** — a set of
specialized agents coordinated by a sequential orchestrator.

## Architecture Overview

```
User Input
    │
    ▼
┌─────────────┐
│ PlannerAgent │  ← Classifies intent (quiz, explain, notes, flashcard...)
└─────┬───────┘
      │
      ▼ (routes to appropriate agent)
┌─────────────────┐   ┌──────────────┐   ┌───────────────┐
│  TeacherAgent   │   │ExaminerAgent │   │ DocumentAgent │
│ (explain/notes) │   │(quiz/flash)  │   │  (ingest)     │
└────────┬────────┘   └──────┬───────┘   └───────┬───────┘
         │                    │                    │
         └────────┬──────────┘────────────────────┘
                  ▼
         ┌──────────────┐
         │ReviewerAgent │  ← Validates output quality
         └──────┬───────┘
                ▼
         ┌─────────────┐
         │ MemoryAgent │  ← Updates progress & spaced repetition
         └─────────────┘
```

Each agent is a thin wrapper around existing NeuroForge workflows, keeping
the system simple while enabling modular orchestration.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

from src.agents import (
    MultiAgentOrchestrator,
    PlannerAgent,
    DocumentAgent,
    TeacherAgent,
    ExaminerAgent,
    ReviewerAgent,
    MemoryAgent,
)
from src.llm import LLMClient
from src.retrieval import Retriever
from src.store import KnowledgeGraph, VectorStore
from src.memory import ProgressTracker, SpacedRepetitionScheduler

print("✓ All agent imports successful")

## Initialize Components

The orchestrator needs all core NeuroForge components. We'll initialize
them with default settings.

In [ ]:
# Initialize core components
llm_client = LLMClient()
vector_store = VectorStore()
knowledge_graph = KnowledgeGraph()
retriever = Retriever(
    vector_store=vector_store,
    knowledge_graph=knowledge_graph,
)
progress_tracker = ProgressTracker(state_file="./demo_learning_state.json")
scheduler = SpacedRepetitionScheduler(state_file="./demo_sr_state.json")

print(f"LLM providers available: {llm_client.available_providers}")
print("✓ All components initialized")

## Individual Agent Demos

Let's test each agent individually before wiring them together.

### PlannerAgent — Intent Classification

In [ ]:
planner = PlannerAgent(llm_client=llm_client)

test_inputs = [
    "Generate 5 easy quiz questions on photosynthesis",
    "Explain machine learning",
    "Make flashcards for chemistry",
    "Give me revision notes on calculus",
    "Hello, how are you?",
]

print("Intent Classification Results:")
print("=" * 60)
for text in test_inputs:
    result = planner.run({"user_input": text})
    print(f"  Input: {text}")
    print(f"  Intent: {result['intent']}")
    print(f"  Params: {result['parameters']}")
    print()

### ReviewerAgent — Quality Validation

In [ ]:
reviewer = ReviewerAgent()

# Test various outputs
test_cases = [
    {"result": "This is a thorough explanation of the concept.", "intent": "explain"},
    {"result": "", "intent": "explain"},
    {"result": None, "intent": "quiz"},
    {"result": [{"q": "What is X?"}], "intent": "quiz"},
    {"result": [], "intent": "flashcard"},
]

print("Quality Validation Results:")
print("=" * 60)
for case in test_cases:
    check = reviewer.run(case)
    status = "✓ PASS" if check["passed"] else "✗ FAIL"
    print(f"  {status} | intent={case['intent']}, result={repr(case['result'])[:40]}")
    if check["issues"]:
        print(f"         Issues: {check['issues']}")
    print()

### MemoryAgent — Progress Tracking

In [ ]:
memory = MemoryAgent(progress_tracker=progress_tracker, scheduler=scheduler)

# Simulate recording a quiz score
result = memory.run({
    "intent": "quiz",
    "topic": "photosynthesis",
    "score": 80.0,
})

print("Memory Update Result:")
print(f"  Updated: {result['updated']}")
print(f"  Topic: {result['topic']}")
print(f"  Mastery: {result['mastery_level']}")
print(f"  Stats: {result['overall_stats']}")

## Full Orchestrator Pipeline

Now let's wire everything together and process requests through the
complete multi-agent pipeline.

In [ ]:
# Initialize the orchestrator
orchestrator = MultiAgentOrchestrator(
    llm_client=llm_client,
    retriever=retriever,
    knowledge_graph=knowledge_graph,
    progress_tracker=progress_tracker,
    scheduler=scheduler,
)

print("✓ MultiAgentOrchestrator initialized")
print(f"  Agents: planner, document, teacher, examiner, reviewer, memory")

In [ ]:
# Process an explanation request
result = orchestrator.process("Explain photosynthesis")

print("Pipeline Result (Explain):")
print("=" * 60)
print(f"  Intent: {result['intent']}")
print(f"  Parameters: {result['parameters']}")
print(f"  Quality Check: {'✓ PASS' if result['quality_check']['passed'] else '✗ FAIL'}")
if result['quality_check']['issues']:
    print(f"  Issues: {result['quality_check']['issues']}")
print(f"  Memory Update: {result.get('memory_update')}")
print()
print("  Agent Result (truncated):")
agent_result = result['result']
if isinstance(agent_result, dict) and 'result' in agent_result:
    content = agent_result['result']
    if isinstance(content, dict) and 'answer' in content:
        print(f"    {content['answer'][:200]}...")
    else:
        print(f"    {str(content)[:200]}...")
else:
    print(f"    {str(agent_result)[:200]}...")

In [ ]:
# Process a quiz generation request
result = orchestrator.process("Generate 3 easy questions on machine learning")

print("Pipeline Result (Quiz):")
print("=" * 60)
print(f"  Intent: {result['intent']}")
print(f"  Parameters: {result['parameters']}")
print(f"  Quality Check: {'✓ PASS' if result['quality_check']['passed'] else '✗ FAIL'}")
print(f"  Memory Updated: {result['memory_update']['updated'] if result['memory_update'] else 'N/A'}")
print()

agent_result = result['result']
if isinstance(agent_result, dict) and agent_result.get('status') == 'success':
    items = agent_result.get('result', [])
    print(f"  Generated {len(items)} quiz questions")
    for i, q in enumerate(items[:3], 1):
        print(f"    {i}. {q.get('question', 'N/A')[:80]}")

## Multi-Agent Collaboration Flow

Let's trace through a complete learning session demonstrating
how the agents collaborate.

In [ ]:
# Simulate a learning session
session_inputs = [
    "Explain neural networks",
    "Generate 3 quiz questions on neural networks",
    "Make flashcards for backpropagation",
]

print("Learning Session Trace:")
print("=" * 60)

for i, user_input in enumerate(session_inputs, 1):
    print(f"\n[Step {i}] User: \"{user_input}\"")
    print("-" * 40)
    
    result = orchestrator.process(user_input)
    
    print(f"  → Intent: {result['intent']}")
    print(f"  → Agent: {result['result'].get('action', 'N/A') if isinstance(result['result'], dict) else 'N/A'}")
    print(f"  → Quality: {'✓' if result['quality_check']['passed'] else '✗'}")
    if result['memory_update']:
        print(f"  → Memory: updated={result['memory_update']['updated']}")

print("\n" + "=" * 60)
print("Session complete!")

## Summary

The multi-agent system provides:

1. **Modular design** — Each agent handles one responsibility
2. **Sequential orchestration** — Simple pipeline without complex state machines
3. **Quality validation** — ReviewerAgent checks all outputs
4. **Progress tracking** — MemoryAgent updates learning state
5. **Extensibility** — New agents can be added by inheriting BaseAgent

The agents are thin wrappers around existing workflows, keeping the system
maintainable while enabling coordinated multi-step interactions.

In [ ]:
# Cleanup demo state files
import os
for f in ["./demo_learning_state.json", "./demo_sr_state.json"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Cleaned up: {f}")